In [12]:
import cvxpy as cp
import numpy as np
from scipy.spatial import ConvexHull
from itertools import combinations, product
import matplotlib.pyplot as plt

from mpl_toolkits.mplot3d import Axes3D
from matplotlib.animation import FuncAnimation, FFMpegWriter
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import matplotlib.cm as cm

import numpy as np
from numpy.linalg import matrix_rank

In [14]:
def is_strongly_normal_hyperrectangle(A, B):
    """
    Check strong normality of (A,B) system whose input set is a hyperrectangle.

    Parameters:
        A (ndarray): State matrix (n x n).
        B (ndarray): Input matrix (n x m).

    Returns:
        controllable (bool): True if system is controllable.
        B_full_rank (bool): True if B has full column rank.
    """
    n = A.shape[0]

    # Build controllability matrix
    C = B
    for i in range(1, n):
        C = np.hstack((C, np.linalg.matrix_power(A, i) @ B))

    # Check controllability
    controllable = (matrix_rank(C) == n)

    # Check if B has full column rank
    B_full_rank = (matrix_rank(B) == B.shape[1])

    if controllable and B_full_rank:
        strongly_normal = True
    else:
        strongly_normal = False

    return strongly_normal


# Example 1: DC Motor

In [ ]:
N = 1500            # number of time steps
t0 = 0.0
tf = 0.45           # final time (seconds)
Delta_t = (tf - t0) / N

J = 0.01 # kg*m^2 Moment of inertia
b = 0.1  # N*m*s Motor viscous friction constant
Ke = 0.01 # V/(rad/s) Motor back EMF constant
Kt = 0.01 # N*m/A Motor torque constant
R = 1.0  # Ohm Armature resistance
L = 0.5  # H Armature inductance

u_max = 5     # V max voltage input

# State-Space model
# State variables: angular velocity (rad/s) and current (A)
A = np.array([
    [-b/J, Kt/J],
    [-Ke/L, R/L]
])
# Input: Voltage (V)
B = np.array([
    [0.0],
    [1/L]
])

# Input set: Hyperrectangle
U = np.array([
    [-u_max],
    [0.0],
    [u_max]
])

strongly_normal = is_strongly_normal_hyperrectangle(A, B)

print("Strongly Normal:", strongly_normal)

In [ ]:
# Initial and target states
x0 = np.array([5, 1.0])  
xf = np.zeros(2)

# Decision variables
x = cp.Variable((2, N + 1))
u = cp.Variable((1, N))

objective = cp.Minimize(Delta_t * cp.sum(cp.abs(u)))

# Constraints
constraints = []
constraints += [x[:, 0] == x0]

for k in range(N):
    # Euler discretization: x_{k+1} = x_k + Delta_t*(A x_k + B u_k)
    constraints += [x[:, k+1] == x[:, k] + Delta_t * (A @ x[:, k] + B.flatten() * u[:, k])]
    # Control bounds
    constraints += [cp.abs(u[:, k]) <= u_max]

# Final state constraint (reach equilibrium)
constraints += [x[:, N] == xf]

# Solve
prob = cp.Problem(objective, constraints)
prob.solve(solver=cp.ECOS, verbose=True)

print(f"Status: {prob.status}")
print(f"Optimal cost (L1 control proxy): {prob.value:.6f}")
print(f"Final state error norm: {np.linalg.norm(x.value[:, -1] - xf):.4e}")

In [ ]:
# Extract results
t = np.linspace(t0, tf, N + 1)
x_val = x.value
u_val = u.value.flatten()

ang_vel = x_val[0, :]
current = x_val[1, :]

save_plots = True   # Set to True if you want to save

# IEEE-style figure settings
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.size": 9,
    "axes.labelsize": 9,
    "legend.fontsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "axes.grid": True,
    "grid.linestyle": ":",
    "grid.linewidth": 0.5,
    "grid.alpha": 0.7,
})

# Time grid
time_grid = np.linspace(t0, tf, N + 1)

# ------------------------
# State plots
# ------------------------
fig, axs = plt.subplots(2, 1, figsize=(3.5, 2.5), sharex=True)

# Angular velocity
axs[0].plot(time_grid, ang_vel, linewidth=2.0, label=r'$\dot{\theta}$')
axs[0].axhline(xf[0], color='r', linestyle='--', linewidth=1.0, label=r'$\dot{\theta}_{f}$')
axs[0].set_ylabel(r'$\dot{\theta}$ [rad/s]')
axs[0].legend(loc='upper right', frameon=False)
axs[0].set_xlim([0, tf])   # enforce IEEE-style xlim

# Current
axs[1].plot(time_grid, current, linewidth=2.0, label=r'$i$')
axs[1].axhline(xf[1], color='r', linestyle='--', linewidth=1.0, label=r'$i_{f}$')
axs[1].set_ylabel(r'$i$ [A]')
axs[1].set_xlabel(r'Time [s]')
axs[1].legend(loc='upper right', frameon=False)
axs[1].set_xlim([0, tf])   # enforce IEEE-style xlim

plt.tight_layout(pad=0.5)
if save_plots:
    plt.savefig("dc_motor_states.pdf", bbox_inches='tight')
plt.show()

# ------------------------
# Control input plot
# ------------------------
plt.figure(figsize=(3.5, 1.5))
plt.step(time_grid[:-1], u_val, where='post', linewidth=2.0, label=r'$u$')
plt.ylabel(r'Voltage [V]')
plt.xlabel(r'Time [s]')
# plt.legend(loc='upper right', frameon=False)
plt.xlim([0, tf])   # enforce IEEE-style xlim
plt.tight_layout(pad=0.5)
if save_plots:
    plt.savefig("motor_control.pdf", bbox_inches='tight')
plt.show()

# Example 2: LEO Rendezvous

In [22]:
# Spacecraft parameters
m = 100.0                 # mass (kg)
F_thrust = 5          # thrust (N)
u_max = F_thrust / m  # max control acceleration (m/s^2)

# System matrices (example Clohessy-Wiltshire-Hill)
mu = 3.986 * 10**14  # gravitational parameter (m^3/s^2)
R = 7102.8 * 10**3     # Orbital radious of chief spacecraft (m)
n = np.sqrt(mu / R**3)  # orbital rate (rad/s)

# Clohessy-Wiltshire-Hill equations of motion
A = np.array([
    [0, 0, 0, 1, 0, 0],
    [0, 0, 0, 0, 1, 0],
    [0, 0, 0, 0, 0, 1],
    [3*n**2, 0, 0, 0, 2*n, 0],
    [0, 0, 0, -2*n, 0, 0],
    [0, 0, -n**2, 0, 0, 0]
])
B = np.array([
    [0, 0, 0],
    [0, 0, 0],
    [0, 0, 0],
    [1, 0, 0],
    [0, 1, 0],
    [0, 0, 1]
])

# Input set
U = []

# Standard basis vectors
e = np.eye(3)

# 1) Single actuator at ±u_max
for i in range(3):
    for sign in [-1, 1]:
        U.append(sign * u_max * e[i])

# 2) Two actuators sharing ±u_max/2
for i, j in combinations(range(3), 2):
    for signs in product([-1, 1], repeat=2):
        vec = np.zeros(3)
        vec[i] = signs[0] * u_max / 2
        vec[j] = signs[1] * u_max / 2
        U.append(vec)

# 3) All three actuators sharing ±u_max/3
for signs in product([-1, 1], repeat=3):
    vec = np.array(signs) * u_max / 3
    U.append(vec)

U = np.array(U)

In [ ]:
# Problem discretization parameters
N = 1000                      # number of time steps
t0 = 0.0                     # initial time (seconds)
tf = 200.0                  # final time (seconds)
Delta_t = (tf - t0) / N

# C) Far-field start (zero relative velocity)
x0_C = -100.0   # m
y0_C = -500.0   # m
z0_C = -100.0      # m
vx0_C = 0.0
vy0_C = 0.0
vz0_C = 0.0
x0 = np.array([x0_C, y0_C, z0_C, vx0_C, vy0_C, vz0_C])


xf = np.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0])

# Decision variables
x = cp.Variable((6, N+1))  # states at each timestep
u = cp.Variable((3, N))    # controls at each timestep

# Objective: approximate integral of ||u(t)||_1 over [t0, tf]
objective = cp.Minimize(Delta_t * cp.sum(cp.norm1(u, axis=0)))

# Constraints list
constraints = []

# Initial condition
constraints += [x[:,0] == x0]

# Dynamics constraints (discretized)
for k in range(N):
    constraints += [
        (x[:,k+1] - x[:,k]) == Delta_t * (A @ x[:,k] + B @ u[:,k]),
        cp.norm_inf(u[:,k]) <= u_max  # convex hull of admissible input set
    ]

# Final condition
constraints += [x[:,N] == xf]

# Solve the optimization problem
prob = cp.Problem(objective, constraints)
prob.solve(solver=cp.ECOS, verbose=True)

# Results
print(f"Optimal value (fuel proxy): {prob.value:.4f}")
print(f"Final state error: {np.linalg.norm(x.value[:,-1] - xf):.4e}")

# Compute error
error = np.zeros((6, N+1))
for k in range(N+1):
    error[:,k] = x.value[:,k] - xf

# Compute position and velocity errors
pos_error = error[:3, :]
vel_error = error[3:, :]
# Compute norm of the error
norm_pos_error = np.linalg.norm(pos_error, axis=0)
norm_vel_error = np.linalg.norm(vel_error, axis=0)

In [ ]:
# ------------------------
# Plot Settings
# ------------------------
save_plots = True  # Set True to save
show_title = False
animation = False

# IEEE-style figure settings
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.size": 9,
    "axes.labelsize": 9,
    "legend.fontsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "axes.grid": True,
    "grid.linestyle": ":",
    "grid.linewidth": 0.5,
    "grid.alpha": 0.7,
})

# Time grid
time_grid = np.linspace(t0, tf, N + 1)

# ------------------------
# Plot 1: Spacecraft States
# ------------------------
fig, axs = plt.subplots(2, 1, figsize=(3.5, 3.0), sharex=True)

# Position
position_labels = [r'$r_x$', r'$r_y$', r'$r_z$']
for i in range(3):
    axs[0].plot(time_grid, x.value[i, :], linewidth=1.5, label=position_labels[i])
axs[0].set_ylabel(r'Position [m]')
axs[0].axhline(xf[0], color='r', linestyle='--', linewidth=1.0, label=r'$r_{f}$')
axs[0].legend(loc='lower right', frameon=False)
axs[0].set_xlim([0, tf])

# Velocity
velocity_labels = [r'$v_x$', r'$v_y$', r'$v_z$']
for i in range(3, 6):
    axs[1].plot(time_grid, x.value[i, :], linewidth=1.5, label=velocity_labels[i - 3])
axs[1].set_xlabel(r'Time [s]')
axs[1].set_ylabel(r'Velocity [m/s]')
axs[1].axhline(xf[0], color='r', linestyle='--', linewidth=1.0, label=r'$v_{f}$')
axs[1].legend(loc='upper left', frameon=False)
axs[1].set_xlim([0, tf])

plt.tight_layout(pad=0.5)
if save_plots:
    plt.savefig("spacecraft_states.pdf", bbox_inches='tight')
plt.show()

# ------------------------
# Plot 2: Optimal Control Inputs
# ------------------------
plt.figure(figsize=(3.5, 2.0))
control_labels = [r'$u_x$', r'$u_y$', r'$u_z$']
for i in range(3):
    plt.step(time_grid[:-1], u.value[i, :], where='post', linewidth=1.5, label=control_labels[i])
plt.xlabel(r'Time [s]')
plt.ylabel(r'Control input [$\frac{m}{s^2}$]')
plt.legend(loc='upper right', frameon=False)
plt.xlim([0, tf])
plt.tight_layout(pad=0.5)
if save_plots:
    plt.savefig("optimal_control_inputs.pdf", bbox_inches='tight')
plt.show()

# ------------------------
# Plot 4: 3D Trajectory
# ------------------------
fig = plt.figure(figsize=(3.5, 3.0))
ax = fig.add_subplot(111, projection='3d')
ax.plot(x.value[0, :], x.value[1, :], x.value[2, :], linewidth=1.5, label=r'Trajectory', color='blue')
ax.scatter(x.value[0, 0], x.value[1, 0], x.value[2, 0], color='green', s=20, label=r'Initial Position', marker='o')
ax.scatter(x.value[0, -1], x.value[1, -1], x.value[2, -1], color='red', s=20, label=r'Final Position', marker='x')

ax.set_xlabel(r'$r_x$ [m]')
ax.set_ylabel(r'$r_y$ [m]')
ax.set_zlabel(r'$r_z$ [m]')
ax.legend(loc='best', fontsize=7, frameon=False)

# Equal aspect ratio
max_range = np.array([
    x.value[0,:].max()-x.value[0,:].min(), 
    x.value[1,:].max()-x.value[1,:].min(), 
    x.value[2,:].max()-x.value[2,:].min()
]).max() / 2.0
mid_x = (x.value[0,:].max()+x.value[0,:].min()) * 0.5
mid_y = (x.value[1,:].max()+x.value[1,:].min()) * 0.5
mid_z = (x.value[2,:].max()+x.value[2,:].min()) * 0.5
ax.set_xlim(mid_x - max_range, mid_x + max_range)
ax.set_ylim(mid_y - max_range, mid_y + max_range)
ax.set_zlim(mid_z - max_range, mid_z + max_range)

plt.tight_layout(pad=0.5)
if save_plots:
    fig.savefig("trajectory_3d.pdf", bbox_inches='tight')
plt.show()